In [1]:
import pandas as pd
import numpy as np

In [4]:
# ─────────────────────────────────────────
# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# ─────────────────────────────────────────
df = pd.read_csv("../data/processed/almaty_pm25_matrix.csv", parse_dates=["datetime_utc"])

# Переводим в long-формат: datetime | station_id | pm25
df_long = df.melt(
    id_vars="datetime_utc",
    var_name="station_id",
    value_name="pm25"
)


In [5]:
df.head()

,datetime_utc,2812561,2812563,2812565,2812576,2812620,2812632,2812635,2812649,2812651,...,2812751,2812753,2812769,2812775,2812784,2812792,2812809,2812826,2812831,2812833
0,2024-04-26 10:00:00+00:00,1.985000,9.085000,5.747500,4.860000,9.433333,5.456667,10.902500,1973.367500,7.393333,...,11.620000,12.500000,11.432500,9.305000,7.433333,10.147500,7.173333,10.323333,2.5600,13.585000
1,2024-04-26 11:00:00+00:00,2.996667,5.956667,10.733333,5.635000,3.340000,9.470000,8.990000,1981.046667,6.960000,...,4.663333,1.660000,5.923333,1.423333,3.692500,6.786667,7.875000,10.107500,5.1625,5.593333
2,2024-04-26 12:00:00+00:00,5.052500,6.207500,8.912500,5.763333,9.283333,8.176667,7.856667,1995.300000,10.410000,...,1.622500,0.245000,3.072500,3.133333,0.650000,7.077500,4.320000,12.686667,10.6600,4.825000
3,2024-04-26 13:00:00+00:00,7.790000,5.886667,12.436667,10.552500,4.407500,10.462500,6.280000,2006.806667,11.876667,...,4.153333,2.583333,3.313333,6.723333,2.660000,10.093333,7.763333,8.333333,6.2800,9.923333
4,2024-04-26 14:00:00+00:00,8.730000,10.170000,14.020000,10.683333,3.490556,10.715000,7.770000,2021.990000,16.220000,...,7.530000,4.330000,7.590000,4.440000,3.890000,11.356667,8.990000,18.950000,6.9300,9.757778


In [6]:
# Дополнительные временные колонки
df_long["year"]  = df_long["datetime_utc"].dt.year
df_long["month"] = df_long["datetime_utc"].dt.month
df_long["hour"]  = df_long["datetime_utc"].dt.hour
df_long["month_name"] = df_long["datetime_utc"].dt.strftime("%Y-%m")

In [7]:
print("=" * 65)
print("  АНАЛИЗ ДАННЫХ PM2.5 — АЛМАТЫ")
print("=" * 65)
print(f"\nПериод данных : {df_long['datetime_utc'].min()} → {df_long['datetime_utc'].max()}")
print(f"Кол-во станций: {df_long['station_id'].nunique()}")
print(f"Кол-во записей: {len(df_long):,}")

  АНАЛИЗ ДАННЫХ PM2.5 — АЛМАТЫ

Период данных : 2024-04-26 10:00:00+00:00 → 2024-12-20 23:00:00+00:00
Кол-во станций: 29
Кол-во записей: 166,054


In [8]:
# ─────────────────────────────────────────
# 2. ОБЩАЯ СТАТИСТИКА (все станции вместе)
# ─────────────────────────────────────────
print("\n" + "─" * 65)
print("  ОБЩАЯ СТАТИСТИКА (все станции)")
print("─" * 65)

clean = df_long["pm25"].replace(0, np.nan).dropna()   # 0 обычно = нет данных

stats_global = {
    "Максимум"         : clean.max(),
    "Минимум"          : clean.min(),
    "Среднее"          : clean.mean(),
    "Медиана"          : clean.median(),
    "Стд. отклонение"  : clean.std(),
    "Кол-во валидных"  : int(clean.count()),
}
for k, v in stats_global.items():
    if isinstance(v, float):
        print(f"  {k:<22}: {v:.2f}")
    else:
        print(f"  {k:<22}: {v:,}")


─────────────────────────────────────────────────────────────────
  ОБЩАЯ СТАТИСТИКА (все станции)
─────────────────────────────────────────────────────────────────
  Максимум              : 3757.23
  Минимум               : 0.00
  Среднее               : 166.88
  Медиана               : 14.13
  Стд. отклонение       : 521.74
  Кол-во валидных       : 162,374


In [9]:
# ─────────────────────────────────────────
# 3. СРЕДНЕЕ PM2.5 ПО МЕСЯЦАМ
# ─────────────────────────────────────────
print("\n" + "─" * 65)
print("  СРЕДНЕЕ PM2.5 ПО МЕСЯЦАМ")
print("─" * 65)

monthly = (
    df_long
    .replace({"pm25": {0: np.nan}})
    .groupby("month_name")["pm25"]
    .agg(
        mean_pm25  = "mean",
        max_pm25   = "max",
        min_pm25   = "min",
        std_pm25   = "std",
        count      = "count",
    )
    .reset_index()
    .rename(columns={"month_name": "Месяц"})
)

print(monthly.to_string(
    index=False,
    float_format=lambda x: f"{x:.2f}",
    col_space=12
))


─────────────────────────────────────────────────────────────────
  СРЕДНЕЕ PM2.5 ПО МЕСЯЦАМ
─────────────────────────────────────────────────────────────────
       Месяц    mean_pm25     max_pm25     min_pm25     std_pm25        count
     2024-04       158.91      2478.05         0.01       506.41         3153
     2024-05       154.58      2421.65         0.00       461.16        20565
     2024-06       126.82      2090.66         0.02       404.53        20869
     2024-07       135.20      1880.01         0.00       413.22        21257
     2024-08       127.68      1986.71         0.02       403.10        21526
     2024-09       160.53      2744.25         0.00       487.59        20492
     2024-10       178.50      2982.70         0.00       536.26        20944
     2024-11       215.02      3382.93         0.00       639.19        20220
     2024-12       282.60      3757.23         0.00       821.21        13348


In [10]:
# ─────────────────────────────────────────
# 4. СТАТИСТИКА ПО СТАНЦИЯМ
# ─────────────────────────────────────────
print("\n" + "─" * 65)
print("  СТАТИСТИКА ПО СТАНЦИЯМ")
print("─" * 65)

by_station = (
    df_long
    .replace({"pm25": {0: np.nan}})
    .groupby("station_id")["pm25"]
    .agg(
        mean_pm25  = "mean",
        max_pm25   = "max",
        min_pm25   = "min",
        std_pm25   = "std",
        count      = "count",
    )
    .reset_index()
    .sort_values("mean_pm25", ascending=False)
)

print(by_station.to_string(
    index=False,
    float_format=lambda x: f"{x:.2f}",
    col_space=12
))


─────────────────────────────────────────────────────────────────
  СТАТИСТИКА ПО СТАНЦИЯМ
─────────────────────────────────────────────────────────────────
  station_id    mean_pm25     max_pm25     min_pm25     std_pm25        count
     2812676      1988.32      3757.23      1407.58       550.99         5726
     2812831      1105.41      3671.07         0.00      1121.22         5682
     2812649       698.11      2386.10         0.04       758.85         5688
     2812716       407.03      2135.61         0.01       624.78         5631
     2812691       113.65      2807.47         0.01       398.85         5661
     2812651        30.52       395.16         0.02        36.18         5676
     2812689        25.93       309.90         0.01        33.00         5683
     2812565        25.64       427.09         0.01        32.45         5655
     2812792        25.32       274.02         0.08        30.60         5672
     2812632        24.90       335.78         0.06        29.

In [11]:
# ─────────────────────────────────────────
# 5. ТОП-5 САМЫХ ГРЯЗНЫХ СТАНЦИЙ
# ─────────────────────────────────────────
print("\n" + "─" * 65)
print("  ТОП-5 СТАНЦИЙ ПО СРЕДНЕМУ PM2.5")
print("─" * 65)
print(by_station.head(5).to_string(index=False, float_format=lambda x: f"{x:.2f}"))


─────────────────────────────────────────────────────────────────
  ТОП-5 СТАНЦИЙ ПО СРЕДНЕМУ PM2.5
─────────────────────────────────────────────────────────────────
station_id  mean_pm25  max_pm25  min_pm25  std_pm25  count
   2812676    1988.32   3757.23   1407.58    550.99   5726
   2812831    1105.41   3671.07      0.00   1121.22   5682
   2812649     698.11   2386.10      0.04    758.85   5688
   2812716     407.03   2135.61      0.01    624.78   5631
   2812691     113.65   2807.47      0.01    398.85   5661


In [12]:
# ─────────────────────────────────────────
# 6. СРЕДНЕЕ ПО СТАНЦИЯМ И ПО МЕСЯЦАМ (pivot)
# ─────────────────────────────────────────
print("\n" + "─" * 65)
print("  СРЕДНЕЕ PM2.5: СТАНЦИЯ × МЕСЯЦ (pivot-таблица)")
print("─" * 65)

pivot = (
    df_long
    .replace({"pm25": {0: np.nan}})
    .groupby(["station_id", "month_name"])["pm25"]
    .mean()
    .unstack("month_name")
    .round(2)
)
print(pivot.to_string())



─────────────────────────────────────────────────────────────────
  СРЕДНЕЕ PM2.5: СТАНЦИЯ × МЕСЯЦ (pivot-таблица)
─────────────────────────────────────────────────────────────────
month_name  2024-04  2024-05  2024-06  2024-07  2024-08  2024-09  2024-10  2024-11  2024-12
station_id                                                                                 
2812561       11.77    17.92    20.70    20.70    20.70    20.70    20.70    20.70    20.70
2812563       12.50    11.36    11.83     9.58    12.34    16.05    29.22    56.49    50.02
2812565       14.91    14.06    14.91    12.54    14.85    20.80    33.02    59.86    44.96
2812576       17.02    12.74    13.09    10.78    13.58    17.00    27.47    43.82    44.81
2812620       13.39     9.46    10.69     9.65    11.18    10.87    14.41    20.89    22.87
2812632       16.35    15.17    15.93    13.24    15.20    22.26    31.56    55.91    36.50
2812635       13.83    10.10    11.66    10.01    11.63    14.35    22.34    39.36